In [ ]:
# 0) Install DuckDB (fast + low-RAM joins)
!pip -q install duckdb


In [ ]:
# 1) Mount Google Drive (recommended for large files)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 2) Set file paths (EDIT THESE)
# Put your two CSVs in Google Drive and update the paths below.
RECIPES = "/content/RAW_recipes.csv"
INTERACTIONS = "/content/RAW_interactions.csv"


In [ ]:
import os; print(os.path.exists(RECIPES), os.path.exists(INTERACTIONS))

True True


In [ ]:
OUT_PARQUET = "/content/drive/MyDrive/master.parquet"
OUT_CSV = "/content/drive/MyDrive/master.csv"


In [ ]:
import os
print(os.path.exists(OUT_PARQUET))


False


In [ ]:
# =========================
# PHASE 1: Preprocessing + Join (Colab End-to-End)
# Two big CSVs (~350MB each) -> one clean master dataset (Parquet)
# Join key: interactions.recipe_id  ->  recipes.id
# Output: master.parquet + data quality summary
# NOTE: Your CSV columns "submitted" and "date" are already detected as DATE by DuckDB,
#       so we DO NOT use TRY_STRPTIME here.
# =========================



# 5) Read CSVs into DuckDB tables
con.execute(f"""
CREATE OR REPLACE TABLE recipes_raw AS
SELECT * FROM read_csv_auto('{RECIPES}', sample_size=200000);
""")

con.execute(f"""
CREATE OR REPLACE TABLE interactions_raw AS
SELECT * FROM read_csv_auto('{INTERACTIONS}', sample_size=200000);
""")

# 6) Inspect schemas (helps confirm types)
print("\n=== recipes_raw schema ===")
print(con.execute("DESCRIBE recipes_raw").fetchdf())

print("\n=== interactions_raw schema ===")
print(con.execute("DESCRIBE interactions_raw").fetchdf())

# 7) Clean + normalize
# - Align join key name to "recipe_id"
# - Cast types consistently
# - Keep dates as DATE (DuckDB already inferred them)
con.execute("""
CREATE OR REPLACE TABLE recipes_clean AS
SELECT
  CAST(id AS BIGINT) AS recipe_id,
  name,
  CAST(minutes AS BIGINT) AS minutes,
  CAST(contributor_id AS BIGINT) AS contributor_id,
  CAST(submitted AS DATE) AS submitted_date,
  tags,
  nutrition,
  CAST(n_steps AS BIGINT) AS n_steps,
  steps,
  description,
  ingredients,
  CAST(n_ingredients AS BIGINT) AS n_ingredients
FROM recipes_raw;
""")

con.execute("""
CREATE OR REPLACE TABLE interactions_clean AS
SELECT
  CAST(user_id AS BIGINT) AS user_id,
  CAST(recipe_id AS BIGINT) AS recipe_id,
  CAST(date AS DATE) AS interaction_date,
  CAST(rating AS DOUBLE) AS rating,
  review
FROM interactions_raw;
""")

# 8) Pre-join quality checks
recipes_total = con.execute("SELECT COUNT(*) FROM recipes_clean").fetchone()[0]
interactions_total = con.execute("SELECT COUNT(*) FROM interactions_clean").fetchone()[0]

recipes_dup_keys = con.execute("""
SELECT COUNT(*) FROM (
  SELECT recipe_id
  FROM recipes_clean
  GROUP BY recipe_id
  HAVING COUNT(*) > 1
);
""").fetchone()[0]

interactions_missing_key = con.execute("""
SELECT COUNT(*) FROM interactions_clean WHERE recipe_id IS NULL;
""").fetchone()[0]

print("\n=== Pre-Join Quality Summary ===")
print("recipes_total:", recipes_total)
print("interactions_total:", interactions_total)
print("recipes_duplicate_recipe_id_count:", recipes_dup_keys)
print("interactions_missing_recipe_id:", interactions_missing_key)

# 9) Build MASTER dataset (LEFT JOIN keeps all interactions)
con.execute("""
CREATE OR REPLACE TABLE master AS
SELECT
  i.user_id,
  i.recipe_id,
  i.interaction_date,
  i.rating,
  i.review,

  r.name,
  r.minutes,
  r.contributor_id,
  r.submitted_date,
  r.tags,
  r.nutrition,
  r.n_steps,
  r.steps,
  r.description,
  r.ingredients,
  r.n_ingredients
FROM interactions_clean i
LEFT JOIN recipes_clean r
USING (recipe_id);
""")

# 10) Post-join checks
master_total = con.execute("SELECT COUNT(*) FROM master").fetchone()[0]
missing_recipe_matches = con.execute("""
SELECT COUNT(*) FROM master WHERE name IS NULL;
""").fetchone()[0]

match_rate = (1 - (missing_recipe_matches / master_total)) * 100 if master_total else 0

print("\n=== Post-Join Quality Summary ===")
print("master_total (should match interactions_total):", master_total)
print("missing_recipe_matches (name is NULL):", missing_recipe_matches)
print("match_rate:", round(match_rate, 2), "%")

# 11) Export to Parquet (recommended)
con.execute(f"COPY master TO '{OUT_PARQUET}' (FORMAT PARQUET);")
print("\n✅ Wrote Parquet to:", OUT_PARQUET)

# Optional: export CSV (uncomment if needed)
# con.execute(f"COPY master TO '{OUT_CSV}' (HEADER, DELIMITER ',');")
# print("✅ Wrote CSV to:", OUT_CSV)

# 12) Confirm output exists + size
print("\n=== Output File Check ===")
print("Parquet exists:", os.path.exists(OUT_PARQUET))
if os.path.exists(OUT_PARQUET):
    print("Parquet size (MB):", round(os.path.getsize(OUT_PARQUET)/1024/1024, 2))

# 13) Quick sample preview
print("\n=== Sample master rows ===")
print(con.execute("SELECT * FROM master LIMIT 5").fetchdf())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== recipes_raw schema ===
       column_name column_type null   key default extra
0             name     VARCHAR  YES  None    None  None
1               id      BIGINT  YES  None    None  None
2          minutes      BIGINT  YES  None    None  None
3   contributor_id      BIGINT  YES  None    None  None
4        submitted        DATE  YES  None    None  None
5             tags     VARCHAR  YES  None    None  None
6        nutrition     VARCHAR  YES  None    None  None
7          n_steps      BIGINT  YES  None    None  None
8            steps     VARCHAR  YES  None    None  None
9      description     VARCHAR  YES  None    None  None
10     ingredients     VARCHAR  YES  None    None  None
11   n_ingredients      BIGINT  YES  None    None  None

=== interactions_raw schema ===
  column_name column_type null   key default extra
0     user_id      BIGINT  YES  None    None  None
1   recipe_id      BIGINT  YES  None    None  None
2        date        DATE  YES  None    None  None
3      

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== Pre-Join Quality Summary ===
recipes_total: 231637
interactions_total: 1132367
recipes_duplicate_recipe_id_count: 0
interactions_missing_recipe_id: 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== Post-Join Quality Summary ===
master_total (should match interactions_total): 1132367
missing_recipe_matches (name is NULL): 1
match_rate: 100.0 %


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


✅ Wrote Parquet to: /content/drive/MyDrive/master.parquet

=== Output File Check ===
Parquet exists: True
Parquet size (MB): 408.1

=== Sample master rows ===
   user_id  recipe_id interaction_date  rating  \
0    38094      40893       2003-02-17     4.0   
1  1293707      40893       2011-12-21     5.0   
2     8937      44394       2002-12-01     4.0   
3   126440      85009       2010-02-27     5.0   
4    57222      85009       2011-10-01     5.0   

                                              review  \
0  Great with a salad. Cooked on top of stove for...   
1  So simple, so delicious! Great for chilly fall...   
2  This worked very well and is EASY.  I used not...   
3  I made the Mexican topping and took it to bunk...   
4  Made the cheddar bacon topping, adding a sprin...   

                                   name  minutes  contributor_id  \
0  white bean   green chile pepper soup      495            1533   
1  white bean   green chile pepper soup      495            1533  

In [ ]:
import pandas as pd

df = pd.read_parquet("/content/drive/MyDrive/master.parquet")
df.shape, df.columns[:10]


((1132367, 16),
 Index(['user_id', 'recipe_id', 'interaction_date', 'rating', 'review', 'name',
        'minutes', 'contributor_id', 'submitted_date', 'tags'],
       dtype='object'))

In [ ]:
con.execute("""
SELECT
  (SELECT COUNT(*) FROM interactions_clean) AS interactions_rows,
  (SELECT COUNT(*) FROM master) AS master_rows
""").fetchdf()


,interactions_rows,master_rows
0,1132367,1132367


In [ ]:
con.execute("""
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN name IS NULL THEN 1 ELSE 0 END) AS missing_recipe_rows,
  ROUND(100.0 * (1 - SUM(CASE WHEN name IS NULL THEN 1 ELSE 0 END)/COUNT(*)), 2) AS match_rate_pct
FROM master
""").fetchdf()


,total_rows,missing_recipe_rows,match_rate_pct
0,1132367,1.0,100.0


In [ ]:
con.execute("""
SELECT recipe_id, COUNT(*) AS c
FROM recipes_clean
GROUP BY recipe_id
HAVING COUNT(*) > 1
ORDER BY c DESC
LIMIT 20
""").fetchdf()


,recipe_id,c


In [ ]:
con.execute("""
SELECT
  SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END) AS null_user_id,
  SUM(CASE WHEN recipe_id IS NULL THEN 1 ELSE 0 END) AS null_recipe_id,
  SUM(CASE WHEN rating IS NULL THEN 1 ELSE 0 END) AS null_rating,
  SUM(CASE WHEN interaction_date IS NULL THEN 1 ELSE 0 END) AS null_interaction_date
FROM master
""").fetchdf()


,null_user_id,null_recipe_id,null_rating,null_interaction_date
0,0.0,0.0,0.0,0.0


In [ ]:
con.execute("SELECT user_id, recipe_id, rating, name, ingredients, steps FROM master USING SAMPLE 5 ROWS").fetchdf()


,user_id,recipe_id,rating,name,ingredients,steps
0,57222,434181,5.0,quinoa garbanzo spinach salad w smoked pap...,"['quinoa', 'baby spinach leaves', 'garbanzo be...","['place quinoa in large saucepan', 'add enough..."
1,106608,313457,5.0,one pan chocolate snack cake,"['flour', 'sugar', 'salt', 'baking soda', 'bak...","['preheat oven to 350 degrees', 'thoroughly mi..."
2,29956,14569,5.0,strawberry strawberry pie from the really good...,"['pie crust', 'strawberry', 'strawberry jell-o...","['pre-bake pie crust', 'cool', 'slice the stra..."
3,74736,45691,4.0,almost kfc coleslaw,"['cabbage', 'carrot', 'onions', 'granulated su...","['mix cabbage , carrot , and onion in a large ..."
4,129958,252312,5.0,vinegar gravy,"['butter', 'cider vinegar']","['melt the butter in a large frying pan', 'coo..."


In [20]:
OUT_CSV = "/content/drive/MyDrive/master_final_sma.csv"

con.execute(f"COPY master TO '{OUT_CSV}' (HEADER, DELIMITER ',');")
print("CSV exported to:", OUT_CSV)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CSV exported to: /content/drive/MyDrive/master_final_sma.csv
